# Oil Health — Bootstrap Training on Paper Data
Trains KMeans + interpolation splines from Engine 1 (Table 8, Balashanmugam 2016).  
Outputs `centroids.bin` and `interpolation.npz` — copy these to the Raspberry Pi.

In [20]:
# ── Cell 1: Install deps (Kaggle already has sklearn/scipy, just confirm) ──
import importlib, sys
for pkg in ['numpy','pandas','sklearn','scipy']:
    m = importlib.import_module(pkg if pkg != 'sklearn' else 'sklearn')
    print(f"{pkg}: OK")

numpy: OK
pandas: OK
sklearn: OK
scipy: OK


In [21]:
# ── Cell 2: Redirect file paths to /kaggle/working ────────────────────────
import os

WORK = '/kaggle/working'
os.makedirs(f'{WORK}/models', exist_ok=True)
os.makedirs(f'{WORK}/data',   exist_ok=True)

print('Output folder:', WORK)

Output folder: /kaggle/working


In [25]:
# ── Cell 3: Load daq_pipeline and patch paths ─────────────────────────────
# Upload daq_pipeline.py to this notebook via  Add Data → Upload → daq_pipeline.py
# then set the path below.

import importlib.util, sys

PIPELINE_PATH = '/kaggle/input/datasets/krishnasimha/daq-pipeline4/daq_pipeline (8).py'  # adjust if needed

spec = importlib.util.spec_from_file_location('daq_pipeline', PIPELINE_PATH)
daq  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(daq)

# Redirect all file I/O to /kaggle/working
daq.MODEL_PATH    = f'{WORK}/models/centroids.bin'
daq.INTERP_PATH   = f'{WORK}/models/interpolation.npz'
daq.DATA_LOG_PATH = f'{WORK}/data/readings.csv'
daq.LAB_DATA_PATH = f'{WORK}/data/lab_samples.csv'

print('Pipeline loaded. Paths patched to /kaggle/working')

Pipeline loaded. Paths patched to /kaggle/working


In [26]:
# ── Cell 4: Engine 1 paper data (Table 8) ────────────────────────────────
import numpy as np

# Only Engine 1 has all 5 features — used for both KMeans and splines
ENGINE1_HOURS      = [0,     50,    100,   150,   200,   250  ]
ENGINE1_VISCOSITY  = [13.87, 14.27, 14.8,  16.0,  18.65, 31.73]
ENGINE1_TBN        = [10.74, 8.79,  8.95,  7.81,  7.96,  6.21 ]
ENGINE1_TAN        = [3.34,  3.52,  4.04,  3.98,  3.65,  4.36 ]
ENGINE1_DIELECTRIC = [2.2,   2.28,  2.32,  2.5,   2.83,  3.11 ]
ENGINE1_SOOT       = [0.0,   0.87,  2.19,  2.7,   3.44,  4.57 ]

# Stack into feature matrix — column order must match FEATURES = [tbn, tan, visc, dc, soot]
X_real = np.column_stack([
    ENGINE1_TBN,
    ENGINE1_TAN,
    ENGINE1_VISCOSITY,
    ENGINE1_DIELECTRIC,
    ENGINE1_SOOT,
]).astype(np.float32)

print('Feature matrix (6 rows × 5 features):')
import pandas as pd
pd.set_option('display.float_format', '{:.3f}'.format)
print(pd.DataFrame(X_real, columns=daq.FEATURES,
                   index=[f'h={h}' for h in ENGINE1_HOURS]))

Feature matrix (6 rows × 5 features):
         tbn   tan  viscosity  dielectric  soot
h=0   10.740 3.340     13.870       2.200 0.000
h=50   8.790 3.520     14.270       2.280 0.870
h=100  8.950 4.040     14.800       2.320 2.190
h=150  7.810 3.980     16.000       2.500 2.700
h=200  7.960 3.650     18.650       2.830 3.440
h=250  6.210 4.360     31.730       3.110 4.570


In [27]:
# ── Cell 5: Augment — 6 real rows → 30 rows to stabilise KMeans ─────────
# Adds small Gaussian noise (5% of each feature's std).
# Does NOT invent new degradation states — only stabilises cluster geometry.

np.random.seed(42)
noise = X_real.std(axis=0) * 0.05
copies = [X_real + np.random.randn(*X_real.shape) * noise for _ in range(4)]
X_train = np.vstack([X_real] + copies)   # 6 + 24 = 30 rows

print(f'Training matrix after augmentation: {X_train.shape}')
print(f'Feature ranges (min → max):')
for i, f in enumerate(daq.FEATURES):
    print(f'  {f:<12}: {X_train[:,i].min():.3f} → {X_train[:,i].max():.3f}')

Training matrix after augmentation: (30, 5)
Feature ranges (min → max):
  tbn         : 6.176 → 10.774
  tan         : 3.337 → 4.376
  viscosity   : 13.525 → 31.847
  dielectric  : 2.180 → 3.129
  soot        : -0.030 → 4.644


In [28]:
# ── Cell 6: Seed interpolation model with Engine 1 lab samples ───────────

interp  = daq.InterpolationModel().initialise(daq.INTERP_PATH)
lab_log = daq.LabSampleLogger()
lab_log.path = daq.LAB_DATA_PATH

for h, tbn, tan in zip(ENGINE1_HOURS, ENGINE1_TBN, ENGINE1_TAN):
    sample = daq.LabSample(
        engine_hours = h,
        tbn          = tbn,
        tan          = tan,
        engine_id    = 'engine_1',
    )
    interp.add_lab_sample(sample)
    lab_log.log(sample)

print('Knot counts:', interp.knot_counts())
print()
print('Spline estimates (sanity check):')
for h in [0, 75, 125, 175, 225]:
    est = interp.estimate(h)
    print(f'  h={h:>4}h → TBN={est["tbn"]:.3f}  TAN={est["tan"]:.3f}')

[INTERP] Loaded splines from /kaggle/working/models/interpolation.npz
[INTERP] Updated splines with lab sample at 0.0h
[INTERP] Updated splines with lab sample at 50.0h
[INTERP] Updated splines with lab sample at 100.0h
[INTERP] Updated splines with lab sample at 150.0h
[INTERP] Updated splines with lab sample at 200.0h
[INTERP] Updated splines with lab sample at 250.0h
Knot counts: {'tbn': 6, 'tan': 6}

Spline estimates (sanity check):
  h=   0h → TBN=10.740  TAN=3.340
  h=  75h → TBN=8.753  TAN=3.790
  h= 125h → TBN=8.388  TAN=4.115
  h= 175h → TBN=7.975  TAN=3.704
  h= 225h → TBN=6.947  TAN=4.056


In [29]:
# ── Cell 7: Run OfflineTrainer on augmented Engine 1 data ────────────────

trainer = daq.OfflineTrainer(interp_model=interp, lab_logger=lab_log)
diag    = trainer.train(X_train)

if 'error' in diag:
    raise RuntimeError(f'Training failed: {diag["error"]}')

print('── Training diagnostics ──────────────────────────────')
print(f'  Samples      : {diag["n_samples"]}')
print(f'  Optimal K    : {diag["optimal_k"]}')
print(f'  Inertia      : {diag["inertia"]}')
print(f'  Cluster sizes: {diag["cluster_sizes"]}')
print(f'  Cluster lbls : {diag["cluster_labels"]}')
print(f'  PC1 var ratio: {diag["pc1_var_ratio"]}')
print(f'  OHI weights  : {diag["ohi_weights"]}')

[ELBOW]  k=2  inertia=6.3268
[ELBOW]  k=3  inertia=2.3436
[ELBOW]  k=4  inertia=0.9522
[ELBOW]  k=5  inertia=0.3200
[ELBOW]  k=6  inertia=0.0227
[ELBOW]  No clear knee found — falling back to K=3 (physical default)
[OFFLINE] Elbow → optimal K = 3
[MODEL] Exported to /kaggle/working/models/centroids.bin (107 bytes)
[OFFLINE] No lab samples on disk — interpolation model unchanged.
── Training diagnostics ──────────────────────────────
  Samples      : 30
  Optimal K    : 3
  Inertia      : 2.3436
  Cluster sizes: [5, 15, 10]
  Cluster lbls : [2, 0, 1]
  PC1 var ratio: 0.8647
  OHI weights  : {'tbn': 0.1851, 'tan': 0.1832, 'viscosity': 0.207, 'dielectric': 0.2167, 'soot': 0.208}


In [30]:
# ── Cell 8: OHI sanity check on real Engine 1 rows ───────────────────────

print('── OHI on real Engine 1 readings ────────────────────')
for h, visc, tbn, tan, dc, soot in zip(
    ENGINE1_HOURS, ENGINE1_VISCOSITY,
    ENGINE1_TBN, ENGINE1_TAN,
    ENGINE1_DIELECTRIC, ENGINE1_SOOT
):
    reading = daq.SensorReading(
        timestamp    = 0,
        engine_hours = h,
        viscosity    = visc,
        dielectric   = dc,
        soot         = soot,
        tbn          = tbn,
        tan          = tan,
    )
    ohi = trainer.compute_ohi(reading)
    bar = '█' * (ohi // 5) + '░' * (20 - ohi // 5)
    print(f'  h={h:>3}h  OHI={ohi:3d}/100  [{bar}]'
          f'  TBN={tbn:.2f}  Visc={visc:.2f}  Soot={soot:.2f}%')

# Expected: OHI should decrease monotonically from h=0 to h=250

── OHI on real Engine 1 readings ────────────────────
  h=  0h  OHI= 86/100  [█████████████████░░░]  TBN=10.74  Visc=13.87  Soot=0.00%
  h= 50h  OHI= 74/100  [██████████████░░░░░░]  TBN=8.79  Visc=14.27  Soot=0.87%
  h=100h  OHI= 65/100  [█████████████░░░░░░░]  TBN=8.95  Visc=14.80  Soot=2.19%
  h=150h  OHI= 57/100  [███████████░░░░░░░░░]  TBN=7.81  Visc=16.00  Soot=2.70%
  h=200h  OHI= 50/100  [██████████░░░░░░░░░░]  TBN=7.96  Visc=18.65  Soot=3.44%
  h=250h  OHI= 19/100  [███░░░░░░░░░░░░░░░░░]  TBN=6.21  Visc=31.73  Soot=4.57%


In [31]:
# ── Cell 9: Confirm output files exist and check sizes ───────────────────

for fpath in [
    f'{WORK}/models/centroids.bin',
    f'{WORK}/models/interpolation.npz',
    f'{WORK}/data/lab_samples.csv',
]:
    size = os.path.getsize(fpath)
    print(f'  {os.path.basename(fpath):<25} {size:>6} bytes  ✓')

print()
print('Download these two files from the right panel (Output → /kaggle/working/models/):')
print('  centroids.bin      → copy to  /home/pi/oil_health/models/centroids.bin')
print('  interpolation.npz  → copy to  /home/pi/oil_health/models/interpolation.npz')

  centroids.bin                107 bytes  ✓
  interpolation.npz           1182 bytes  ✓
  lab_samples.csv              426 bytes  ✓

Download these two files from the right panel (Output → /kaggle/working/models/):
  centroids.bin      → copy to  /home/pi/oil_health/models/centroids.bin
  interpolation.npz  → copy to  /home/pi/oil_health/models/interpolation.npz


## After downloading the files

Copy both model files to your Raspberry Pi:

```bash
scp centroids.bin      pi@<PI_IP>:/home/pi/oil_health/models/
scp interpolation.npz  pi@<PI_IP>:/home/pi/oil_health/models/
```

Then start the pipeline — it will load these files on startup and skip bootstrapping:

```bash
python daq_pipeline.py --source serial --port /dev/ttyUSB0
```

The splines are seeded from Engine 1's exact TBN/TAN values.  
When your first real lab sample arrives, call `daq.ingest_lab_sample(...)` and the splines will update automatically.

Once you have collected ~50+ real readings, run the full retrain:

```bash
python daq_pipeline.py --retrain
```